In [1]:
import torch
import numpy as np
import pandas as pd

from dice import Dice, DiceFunctions
from functions import (
    MultiheadDiscreteSANetwork,
    DiscreteSANetwork
)

In [2]:
DEVICE = 'cpu'
SEED = 89290

df = pd.read_csv('./../../simulator/recsys-simulator/experiments/dataset_for_dice/log.csv')

In [3]:
sasrec_num = "0"

actions        = torch.load('./data/actions.pt', weights_only=False)
rewards        = torch.load('./data/rewards.pt', weights_only=False)
states         = torch.load(f'./data/sasrec_{0}_states.pt', weights_only=False)
target_actions = torch.load(f'./data/sasrec_{sasrec_num}_actions.pt', weights_only=False)
action_embs    = torch.load(f'./data/sasrec_{0}_action_embs.pt', weights_only=False)

In [4]:
num_items = df['movieid'].unique().shape[0]

state_dim = states[0].shape[1]
action_dim = action_embs.shape[1]

hidden_dim = 256
num_layers = 4
lr = 3.855348946610917e-06

q_func = DiscreteSANetwork(
    state_dim=state_dim,
    hidden_dim=hidden_dim,
    action_dim=action_dim,
    num_layers=num_layers,
    action_emb=action_embs,
    seed=SEED,
    device=DEVICE
)

w_func = DiscreteSANetwork(
    state_dim=state_dim,
    hidden_dim=hidden_dim,
    action_dim=action_dim,
    num_layers=num_layers,
    action_emb=action_embs,
    seed=SEED,
    device=DEVICE
)

dice = Dice(
    q_function=q_func,
    w_function=w_func,
    gamma=0.99,
    q_lr=lr,
    w_lr=lr,
    lambda_lr=lr,
    f1_function=DiceFunctions.DUAL_DICE_P_3_2,
    f2_function=DiceFunctions.CHI_SQUARED,
    method_name='dual_dice',
    seed=SEED,
    device=DEVICE
)

In [5]:
dice.fit(
    state=states,
    action=actions,
    reward=rewards,
    target_action=target_actions,
    num_steps=250000,
    batch_size=8192,
    eval_iter=100,
    num_workers=8,
    result_folder=f'sasrec_0_test'
)

  0%|          | 0/250000 [00:00<?, ?it/s]/Users/anthony/anaconda3/envs/dice/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
loss: -0.0342; value: 0.7414:   0%|          | 919/250000 [01:14<5:09:08, 13.43it/s] libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::s

KeyboardInterrupt: 

In [ ]:
np.array(dice._experiment_data['value_per_step'])[-300:].mean()

np.float64(0.5005304560201216)

In [ ]:
step_reward = dice.predict_per_step_reward(
    state=states, action=actions, reward=rewards
)

traj_reward = dice.predict_per_traj_reward(
    state=states, action=actions, reward=rewards
)

print(f"per step reward: {step_reward:.7f}")
print(f"per trajectory reward: {traj_reward:.7f}")

per step reward: 0.4694986
per trajectory reward: 70.4247873


In [ ]:
w = dice.predict_weights(np.concatenate(states), np.concatenate(actions))

w.min(),w.mean(), w.max()

(np.float32(-2.67038), np.float32(0.61084664), np.float32(3.5714254))